Logistic Regression Attempt
---
Following Prince's Lab from Nov 2nd to Start

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupKFold, GridSearchCV
from sklearn.metrics import log_loss, confusion_matrix, accuracy_score, precision_score, f1_score

from sklearn.base import clone
from sklearn.metrics import roc_curve, roc_auc_score

from sklearn.calibration import calibration_curve


px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = px.colors.qualitative.Dark24

In [2]:
# load in data
df = pd.read_csv('../Loan_default.csv')
df

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255342,8C6S86ESGC,19,37979,210682,541,109,4,14.11,12,0.85,Bachelor's,Full-time,Married,No,No,Other,No,0
255343,98R4KDHNND,32,51953,189899,511,14,2,11.55,24,0.21,High School,Part-time,Divorced,No,No,Home,No,1
255344,XQK1UUUNGP,56,84820,208294,597,70,3,5.29,60,0.50,High School,Self-employed,Married,Yes,Yes,Auto,Yes,0
255345,JAO28CPL4H,42,85109,60575,809,40,1,20.90,48,0.44,High School,Part-time,Single,Yes,Yes,Other,No,0


In [3]:
df.columns # target column = "default"

Index(['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner',
       'Default'],
      dtype='str')

In [4]:
df.dtypes

LoanID                str
Age                 int64
Income              int64
LoanAmount          int64
CreditScore         int64
MonthsEmployed      int64
NumCreditLines      int64
InterestRate      float64
LoanTerm            int64
DTIRatio          float64
Education             str
EmploymentType        str
MaritalStatus         str
HasMortgage           str
HasDependents         str
LoanPurpose           str
HasCoSigner           str
Default             int64
dtype: object

In [5]:
# save the target column in y
y = df['Default']

# save the predictors or features in x
x = df[['Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']]

# save the categorical variables for preprocessing
cats = ['Education', 'EmploymentType','MaritalStatus',
        'HasMortgage', 'HasDependents','LoanPurpose','HasCoSigner']

# save the numerical variables for preprocessing
nums = ['Age', 'Income', 'LoanAmount',   'CreditScore',
        'MonthsEmployed','NumCreditLines','InterestRate', 'LoanTerm',
        'DTIRatio']

In [6]:
# split data into training and test
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=6021, stratify=y
)

In [7]:
# create the order of the pipeline
preprocess = ColumnTransformer(transformers=[
    ("encoder", OneHotEncoder(drop='first'), cats),
    ("numeric", StandardScaler(), nums)
])

In [8]:
# create a log reg model
logreg=LogisticRegression(max_iter=5000, random_state=123)

# create a preporcessing pipeline
pipe = Pipeline(steps=[("preprocess", preprocess), ("model", logreg)])

# fit
pipe.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](16,)","['Age','Income','LoanAmount',...,'HasDependents','LoanPurpose', 'HasCoSigner']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,16
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('encoder', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthr

In [9]:
# get the predicted probabilities
p_train = pipe.predict_proba(x_train)[:,1]
p_test = pipe.predict_proba(x_test)[:,1]

# get the log odds
log_odds_train = pipe.decision_function(x_train)
log_odds_test = pipe.decision_function(x_test)

# starting with 0.5 as threshold
threshold = 0.5

In [10]:
print("Intercept:", logreg.intercept_[0])

coef = pd.DataFrame({
    "Feature": pipe.named_steps["preprocess"].get_feature_names_out(), #x_train.columns
    "Coefficient": logreg.coef_[0],
    "Odds Ratio": np.exp(logreg.coef_[0])
})

coef.sort_values("Coefficient", ascending=False)

Intercept: -2.088967791645449


,Feature,Coefficient,Odds Ratio
21,numeric__InterestRate,0.457489,1.580102
5,encoder__EmploymentType_Unemployed,0.440726,1.553835
17,numeric__LoanAmount,0.299383,1.349026
3,encoder__EmploymentType_Part-time,0.278911,1.321689
4,encoder__EmploymentType_Self-employed,0.226532,1.254243
20,numeric__NumCreditLines,0.098378,1.103379
0,encoder__Education_High School,0.090011,1.094187
23,numeric__DTIRatio,0.056555,1.058185
10,encoder__LoanPurpose_Business,0.052065,1.053444
22,numeric__LoanTerm,0.005044,1.005056


In [11]:
# get the predicted values
y_pred = (p_test >= threshold).astype(int)

# create a confusion matrix
cm = confusion_matrix(y_test, y_pred, labels = [1,0])

# confusion matrix dataframe
cm_df = pd.DataFrame(
    cm,
    index = ["Actual Default", "Actual No Default"],
    columns = ["Predicted Default", 'Predicted No Default']
)

cm_df

,Predicted Default,Predicted No Default
Actual Default,208,5723
Actual No Default,127,45012


In [12]:
fig = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    labels=dict(x="Predicted", y="Actual", color="Count"),
    title="Loan Default Confusion Matrix"
)

fig.update_layout(
    xaxis_title="Predicted Outcome",
    yaxis_title="Actual Outcome"
)

fig.show()

In [13]:
results_test = pd.DataFrame({
    "Set": "Test", "Actual_default": y_test,
    "Predicted_prob_default": p_test,
    "Log_odds": log_odds_test,
    "Predicted_label": (p_test >= threshold).astype(int)
})
results_test.head().round(3)

,Set,Actual_default,Predicted_prob_default,Log_odds,Predicted_label
59805,Test,0,0.040,-3.172,0
13879,Test,0,0.207,-1.345,0
69152,Test,0,0.178,-1.530,0
221284,Test,1,0.499,-0.003,0
45192,Test,0,0.075,-2.507,0


In [14]:
pd.DataFrame({
    "Set": ["Train", "Test"],
    "Log loss": [log_loss(y_train, p_train), log_loss(y_test, p_test)],
    "Accuracy": [accuracy_score(y_train, p_train >= threshold),
                 accuracy_score(y_test, p_test >= threshold)],
    "Precision": [precision_score(y_train, p_train >= threshold),
                  precision_score(y_test, p_test >= threshold)],
    "F1 Score": [f1_score(y_train, p_train >= threshold),
                 f1_score(y_test, y_pred >= threshold)]
})

,Set,Log loss,Accuracy,Precision,F1 Score
0,Train,0.316156,0.885073,0.591486,0.063206
1,Test,0.314763,0.885451,0.620896,0.066390


Testing Different Thresholds
---

In [15]:
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.50, 0.70, 0.90]

# loaning to someone who would not have paid it back is costly,
# so false negatives are costly, we might lean towards a lower
# threshod because of this
cost_fp = 1000 # arbitrary costs for now
cost_fn = 8000

In [16]:
p_test = pipe.predict_proba(x_test)[:,1]

auc = roc_auc_score(y_test, p_test)
logloss = log_loss(y_test, p_test)

test_rows = []

for t in thresholds:
    y_pred = (p_test >= t).astype(int)

    tp, fn, fp, tn = confusion_matrix(y_test, y_pred,
                                      labels = [1,0]).ravel()

    tpr = tp / (tp+fn)
    fpr = fp / (fp+tn)
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn > 0 else 0

    test_rows.append({
        "Threshold": t,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "TPR": tpr, "FPR": fpr,
        "Precision": precision, "Accuracy": accuracy, "F1": f1,
        "Total cost": cost_fp * fp + cost_fn * fn,
        "Log loss": logloss, "ROC AUC": auc
    })

test_results = pd.DataFrame(test_rows)
display(test_results.round(3))

,Threshold,TP,FP,FN,TN,TPR,FPR,Precision,Accuracy,F1,Total cost,Log loss,ROC AUC
0,0.10,4453,17623,1478,27516,0.751,0.390,0.202,0.626,0.318,29447000,0.315,0.75
1,0.15,3437,10083,2494,35056,0.579,0.223,0.254,0.754,0.353,30035000,0.315,0.75
2,0.20,2566,5758,3365,39381,0.433,0.128,0.308,0.821,0.360,32678000,0.315,0.75
3,0.25,1838,3296,4093,41843,0.310,0.073,0.358,0.855,0.332,36040000,0.315,0.75
4,0.30,1282,1883,4649,43256,0.216,0.042,0.405,0.872,0.282,39075000,0.315,0.75
5,0.50,208,127,5723,45012,0.035,0.003,0.621,0.885,0.066,45911000,0.315,0.75
6,0.70,4,1,5927,45138,0.001,0.000,0.800,0.884,0.001,47417000,0.315,0.75
7,0.90,0,0,5931,45139,0.000,0.000,0.000,0.884,0.000,47448000,0.315,0.75


Looking for the threshold that lowers our false negatives because those are more costly. It looks like the lowest threshold, 0.1 decreaes the false negatives.

In [17]:
fpr_curve, tpr_curve, roc_thresholds = roc_curve(y_test, p_test)

roc_data = pd.DataFrame({
    "FPR": fpr_curve,
    "TPR": tpr_curve
})

fig = px.line(
    roc_data, x="FPR", y="TPR",
    title=f"Credit Loan Default ROC Curve | AUC = {auc:.3f}"
)

fig.add_scatter(
    x=test_results["FPR"],
    y=test_results["TPR"],
    mode="markers+text",
    text=[f"t = {t:.1f}" for t in thresholds],
    textposition="bottom right",
    marker=dict(size=10),
    name="Five thresholds"
)

fig.add_scatter(
    x=[0, 1], y=[0, 1],
    mode="lines", line=dict(dash="dash", color="gray"),
    name="Random classification"
)

fig.update_xaxes(range=[0, 1], title="False Positive Rate (FPR)")
fig.update_yaxes(range=[0, 1], title="True Positive Rate (TPR)")
fig.show()

Based off the ROC AUC, the best threshold would be 0.1 because it is the most left

In [18]:
prob_true, prob_pred = calibration_curve(
    y_test, p_test, n_bins=10, strategy="uniform"
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=prob_pred,
    y=prob_true,
    mode="lines+markers",
    name="Credit Loan Default Logistic Regression"
))

fig.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode="lines",
    line=dict(dash="dash"),
    name="Perfect Calibration"
))

fig.update_layout(
    title="Default on Loan Probability Calibration",
    xaxis_title="Mean Predicted Probability",
    yaxis_title="Observed Default Frequency",
    width=700,
    height=500
)

fig.show()

Model is relatively well calibrated when it is on or near the red line, and it underestimating the default risk when the blue line is above the red line.

Regularization
---

In [19]:
loan_id = df["LoanID"]
train_default, test_default, = train_test_split(
    loan_id.unique(), test_size=0.2, random_state=6021
)

train_default, val_default = train_test_split(
    train_default, test_size=0.25, random_state=6021
)



In [20]:
x_train, y_train = x[loan_id.isin(train_default)], y[loan_id.isin(train_default)]
x_val, y_val = x[loan_id.isin(val_default)], y[loan_id.isin(val_default)]
x_test, y_test = x[loan_id.isin(test_default)], y[loan_id.isin(test_default)]
groups_train = loan_id.loc[x_train.index]

display(pd.DataFrame({
    "Set": ["Training", "Validation", "Test"],
    "Individual Entries": [len(train_default), len(val_default), len(test_default)],
    "Rows": [len(x_train), len(x_val), len(x_test)],
    "Default proportion": [y_train.mean(), y_val.mean(), y_test.mean()]
}))

,Set,Individual Entries,Rows,Default proportion
0,Training,153207,153207,0.115536
1,Validation,51070,51070,0.118954
2,Test,51070,51070,0.115077


In [21]:
# rerun the preproccess
preprocess = ColumnTransformer(transformers=[
    ("encoder", OneHotEncoder(drop='first'), cats),
    ("numeric", StandardScaler(), nums)
])

In [22]:
# different alpha levels
C_values = [0.02, 0.05, 0.1, 0.3, 1, 10]
coef_rows = []

for method in ["Ridge (L2)", "Lasso (L1)"]:
    for C in C_values:
        regularized = Pipeline([
            ("preprocess", clone(preprocess)),
            ("model", LogisticRegression(
                l1_ratio=0 if method == "Ridge (L2)" else 1, # L1_ratio = to alpha
                C=C, solver="liblinear", max_iter=5000, tol=1e-8, random_state=123
            ))
        ])
        regularized.fit(x_train, y_train)

        names = regularized.named_steps["preprocess"].get_feature_names_out()
        betas = regularized.named_steps["model"].coef_[0]

        for name, beta in zip(names, betas):
            coef_rows.append({"Method": method, "C": C,
                              "Feature": name.split("__")[-1],
                              "Coefficient": beta})

coef_paths = pd.DataFrame(coef_rows)

for method in ["Ridge (L2)", "Lasso (L1)"]:
    fig = px.line(
        coef_paths[coef_paths["Method"] == method],
        x="C", y="Coefficient", color="Feature", markers=True,
        log_x=True, title=f"Default Loan: {method} Coefficient Paths"
    )
    fig.add_hline(y=0, line_dash="dash")
    fig.show()

The coefficients are not really shrinking or going to 0.

In [23]:
coef_paths["Nonzero"] = coef_paths["Coefficient"].abs() > 1e-6
demo_counts = coef_paths.groupby(["C", "Method"])["Nonzero"].sum().unstack()
display(demo_counts.rename_axis(columns=None))

,Lasso (L1),Ridge (L2)
C,,
0.02,23,24
0.05,23,24
0.10,24,24
0.30,24,24
1.00,24,24
10.00,24,24


In [24]:
# looking at a log reg without regularization
full_model = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", LogisticRegression(
        C=np.inf, solver="lbfgs", max_iter=5000, tol=1e-8, random_state=123 # if C is a really large number, it stands for 0 for lambda, all the other log reg had ridge!
    ))
])

In [25]:
# just a ridge
ridge = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", LogisticRegression(
        l1_ratio=0, solver="liblinear", max_iter=5000, tol=1e-8, random_state=123 # no c here
    ))
])

# just  a lasso
lasso = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", LogisticRegression(
        l1_ratio=1, solver="liblinear", max_iter=5000, tol=1e-8, random_state=123 # no c here
    ))
])

In [26]:
C_grid = np.unique(np.r_[np.logspace(-4, 3, 36), C_values]) # 1. way to do it is to do a c, another way to do it is with log(c)
cv = GroupKFold(n_splits=5, shuffle=True, random_state=6021) # cross-validation

full_model_cv = GridSearchCV(
    ridge, {"model__C": C_grid}, cv=cv, scoring="neg_log_loss", error_score="raise"
)
full_model_cv.fit(x_train, y_train, groups=groups_train)

ridge_cv = GridSearchCV(
    ridge, {"model__C": C_grid}, cv=cv, scoring="neg_log_loss", error_score="raise"
)
ridge_cv.fit(x_train, y_train, groups=groups_train)

lasso_cv = GridSearchCV(
    lasso, {"model__C": C_grid}, cv=cv, scoring="neg_log_loss", error_score="raise"
)
lasso_cv.fit(x_train, y_train, groups=groups_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... tol=1e-08))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.",{'model__C': array([1.0000...00000000e+03])}
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_log_loss'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",GroupKFold(n_... shuffle=True)
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevi

In [27]:
full_model = full_model_cv.best_estimator_
ridge_model = ridge_cv.best_estimator_
lasso_model = lasso_cv.best_estimator_

best_C_full_model = full_model_cv.best_params_["model__C"]
best_C_ridge = ridge_cv.best_params_["model__C"]
best_C_lasso = lasso_cv.best_params_["model__C"]

models = {"All 12 (unpenalized)": full_model,
          "Ridge": ridge_model, "Lasso": lasso_model}
searches = {"All 12 (unpenalized)": full_model_cv,
            "Ridge": ridge_cv,
            "Lasso": lasso_cv}

display(pd.DataFrame({
    "Model": ["All 12 (unpenalized)", "Ridge", "Lasso"],
    "Selected C": [best_C_full_model, best_C_ridge, best_C_lasso],
    "Mean CV log loss": [-full_model_cv.best_score_, -ridge_cv.best_score_, -lasso_cv.best_score_]
}).round(6))

,Model,Selected C,Mean CV log loss
0,All 12 (unpenalized),0.300000,0.313614
1,Ridge,0.300000,0.313614
2,Lasso,0.251189,0.313613


In [28]:
cv_tables = []
for name, search in searches.items():
    results = pd.DataFrame(search.cv_results_)
    cv_tables.append(pd.DataFrame({
        "Model": name,
        "C": results["param_model__C"].astype(float),
        "CV log loss": -results["mean_test_score"],
        "Fold SD": results["std_test_score"]
    }))

cv_results = pd.concat(cv_tables, ignore_index=True)

In [29]:
for name, search in searches.items():
    chosen_C = search.best_params_["model__C"]
    fig = px.line(
        cv_results[cv_results["Model"] == name],
        x="C", y="CV log loss", log_x=True, markers=True,
        title=f"{name}: five-fold CV log loss | selected C = {chosen_C:g}"
    )
    fig.add_vline(x=chosen_C, line_dash="dash", line_color="black")
    fig.update_xaxes(tickvals=[0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
                     ticktext=["0.0001", "0.001", "0.01", "0.1", "1", "10", "100", "1000"])
    fig.show()

In [30]:
import sklearn
from packaging.version import Version
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

C_lasso = 0.01

if Version(sklearn.__version__) >= Version("1.8"):
    lasso_options = {"l1_ratio": 1}
else:
    lasso_options = {"penalty": "l1"}

lasso_model = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", LogisticRegression(
        C=C_lasso,
        solver="liblinear",
        max_iter=5000,
        tol=1e-8,
        random_state=123,
        **lasso_options
    ))
])

lasso_model.fit(x_train, y_train)

models["Lasso"] = lasso_model

print("scikit-learn version:", sklearn.__version__)
print("Lasso C:", lasso_model.named_steps["model"].C)

scikit-learn version: 1.9.0
Lasso C: 0.01


In [31]:
feature_names = full_model.named_steps["preprocess"].get_feature_names_out()
feature_names = [name.split("__")[-1] for name in feature_names]

coefficients = pd.DataFrame({"Feature": feature_names})
model_rows = []
for name, model in models.items():
    beta = model.named_steps["model"].coef_[0]
    coefficients[name] = beta
    model_rows.append({"Model": name, "Nonzero slopes": int((abs(beta) > 1e-6).sum())})

feature_counts = pd.DataFrame(model_rows)
display(coefficients.round(6))
display(feature_counts)
print("Lasso drops:", coefficients.loc[abs(coefficients["Lasso"]) <= 1e-6, "Feature"].tolist())

,Feature,All 12 (unpenalized),Ridge,Lasso
0,Education_High School,0.069350,0.069350,0.044604
1,Education_Master's,-0.138643,-0.138643,-0.105616
2,Education_PhD,-0.175199,-0.175199,-0.141436
3,EmploymentType_Part-time,0.288426,0.288426,0.134138
4,EmploymentType_Self-employed,0.233533,0.233533,0.078494
5,EmploymentType_Unemployed,0.446218,0.446218,0.293509
6,MaritalStatus_Married,-0.250689,-0.250689,-0.203065
7,MaritalStatus_Single,-0.058348,-0.058348,-0.016054
8,HasMortgage_Yes,-0.153282,-0.153282,-0.136148
9,HasDependents_Yes,-0.269896,-0.269896,-0.250790


,Model,Nonzero slopes
0,All 12 (unpenalized),24
1,Ridge,24
2,Lasso,21


Lasso drops: ['LoanPurpose_Education', 'LoanPurpose_Other', 'LoanTerm']


In [32]:
C_path = np.unique(np.r_[C_grid, C_values])
path_rows = []

for name, search in searches.items():
    for C in C_path:
        path_model = clone(search.best_estimator_).set_params(model__C=C)
        path_model.fit(x_train, y_train)
        beta = path_model.named_steps["model"].coef_[0]
        for feature, value in zip(feature_names, beta):
            path_rows.append({"Model": name, "C": C,
                              "Feature": feature, "Coefficient": value})

paths = pd.DataFrame(path_rows)

In [33]:
paths

,Model,C,Feature,Coefficient
0,All 12 (unpenalized),0.0001,Education_High School,-0.138903
1,All 12 (unpenalized),0.0001,Education_Master's,-0.186324
2,All 12 (unpenalized),0.0001,Education_PhD,-0.194392
3,All 12 (unpenalized),0.0001,EmploymentType_Part-time,-0.152572
4,All 12 (unpenalized),0.0001,EmploymentType_Self-employed,-0.165191
...,...,...,...,...
2803,Lasso,1000.0000,MonthsEmployed,-0.351437
2804,Lasso,1000.0000,NumCreditLines,0.089476
2805,Lasso,1000.0000,InterestRate,0.462084
2806,Lasso,1000.0000,LoanTerm,0.006161


In [34]:
for name, search in searches.items():
    chosen_C = search.best_params_["model__C"]
    fig = px.line(
        paths[paths["Model"] == name], x="C", y="Coefficient",
        color="Feature", log_x=True,
        category_orders={"Feature": feature_names},
        title=f"{name}: coefficient paths | CV-selected C = {chosen_C:g}"
    )
    fig.add_hline(y=0, line_dash="dot", line_color="gray")
    fig.add_vline(x=chosen_C, line_dash="dash", line_color="black")
    fig.update_layout(height=500)
    fig.update_xaxes(tickvals=[0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
                     ticktext=["0.0001", "0.001", "0.01", "0.1", "1", "10", "100", "1000"])
    fig.show()

In [35]:
sparsity_rows = []
for C in [0.01, 0.02, best_C_lasso]:
    demo_lasso = clone(lasso_model).set_params(model__C=C).fit(x_train, y_train)
    beta = demo_lasso.named_steps["model"].coef_[0]
    grid_match = cv_results[(cv_results["Model"] == "Lasso") & np.isclose(cv_results["C"], C)]
    cv_loss = grid_match["CV log loss"].iloc[0] if len(grid_match) else np.nan
    sparsity_rows.append({"C": C, "Nonzero slopes": int((abs(beta) > 1e-6).sum()),
                          "Mean CV log loss": cv_loss,
                          "Role": "CV optimum" if C == best_C_lasso else "Stronger-penalty example"})

display(pd.DataFrame(sparsity_rows).round(6))

,C,Nonzero slopes,Mean CV log loss,Role
0,0.010000,21,0.314065,Stronger-penalty example
1,0.020000,23,0.313726,Stronger-penalty example
2,0.251189,24,0.313613,CV optimum


In [36]:
cost_fp, cost_fn = 3, 1
thresholds = np.round(np.arange(0.00, 1.01, 0.01), 2)
p_val = {name: model.predict_proba(x_val)[:, 1] for name, model in models.items()}

threshold_rows = []
for name, probabilities in p_val.items():
    for t in thresholds:
        predicted = (probabilities >= t).astype(int)
        tp, fn, fp, tn = confusion_matrix(y_val, predicted, labels=[1, 0]).ravel()
        threshold_rows.append({"Model": name, "Threshold": t,
                               "Validation cost": cost_fp * fp + cost_fn * fn,
                               "Validation accuracy": (tp + tn) / len(y_val)})

validation_results = pd.DataFrame(threshold_rows)

In [37]:
validation_results

,Model,Threshold,Validation cost,Validation accuracy
0,All 12 (unpenalized),0.00,134985,0.118954
1,All 12 (unpenalized),0.01,132911,0.132348
2,All 12 (unpenalized),0.02,123868,0.190679
3,All 12 (unpenalized),0.03,112934,0.260427
4,All 12 (unpenalized),0.04,101801,0.330860
...,...,...,...,...
298,Lasso,0.96,6075,0.881046
299,Lasso,0.97,6075,0.881046
300,Lasso,0.98,6075,0.881046
301,Lasso,0.99,6075,0.881046


In [38]:
best_thresholds = {}
selected_rows = []

for name in models:
    results = validation_results[validation_results["Model"] == name].copy()
    results["Distance from 0.5"] = abs(results["Threshold"] - 0.5)
    best = results.sort_values(["Validation cost", "Distance from 0.5", "Threshold"]).iloc[0]
    best_thresholds[name] = float(best["Threshold"])
    selected_rows.append(best)

threshold_summary = pd.DataFrame(selected_rows).drop(columns="Distance from 0.5")
display(threshold_summary.round(4))

,Model,Threshold,Validation cost,Validation accuracy
78,All 12 (unpenalized),0.78,6075,0.881
179,Ridge,0.78,6075,0.881
276,Lasso,0.74,6075,0.881


In [39]:
fig = px.line(
    validation_results, x="Threshold", y="Validation cost", color="Model",
    title="Choose thresholds on validation data: cost = 3 × FP + FN"
)
for name, t in best_thresholds.items():
    point = threshold_summary[threshold_summary["Model"] == name]
    fig.add_scatter(x=[t], y=point["Validation cost"], mode="markers",
                    marker=dict(size=11, symbol="diamond"),
                    name=f"{name}: selected t = {t:.2f}")
fig.show()

In [40]:
p_test = {name: model.predict_proba(x_test)[:, 1] for name, model in models.items()}

test_comparison = pd.DataFrame({
    "Model": list(models),
    "Test log loss": [log_loss(y_test, p_test[name]) for name in models],
    "Nonzero slopes": feature_counts["Nonzero slopes"],
    "Selected threshold": [best_thresholds[name] for name in models]
})
display(test_comparison.round(6))

,Model,Test log loss,Nonzero slopes,Selected threshold
0,All 12 (unpenalized),0.315039,24,0.78
1,Ridge,0.315039,24,0.78
2,Lasso,0.315308,21,0.74


In [41]:
logloss_figure = px.bar(
    test_comparison, x="Model", y="Test log loss", text_auto=".4f",
    title="Test log loss — lower is better"
)
logloss_figure.update_traces(marker_color="#234C6A", width=0.55, textposition="outside")
logloss_figure.update_layout(showlegend=False, xaxis_title=None, height=420)
logloss_figure.update_xaxes(showgrid=False)
logloss_figure.update_yaxes(showgrid=False, range=[0, test_comparison["Test log loss"].max() * 1.16])
logloss_figure.show()

In [42]:
model_names = list(models)
labels = ["Win", "Loss"]
colors = [[0, "#F6F9FC"], [1, "#9EC1DF"]]
titles = ([f"{name}<br>t = 0.50" for name in model_names] +
          [f"{name}<br>selected t = {best_thresholds[name]:.2f}" for name in model_names])
selected_cm_figure = make_subplots(rows=2, cols=3, subplot_titles=titles,
                                  vertical_spacing=0.25, horizontal_spacing=0.10)

for row in [1, 2]:
    for col, name in enumerate(model_names, start=1):
        t = 0.50 if row == 1 else best_thresholds[name]
        cm = confusion_matrix(y_test, p_test[name] >= t, labels=[1, 0])
        selected_cm_figure.add_trace(go.Heatmap(
            z=cm, x=labels, y=labels, zmin=0, zmax=int(y_test.value_counts().max()),
            colorscale=colors, showscale=False, texttemplate="%{z}",
            textfont=dict(color="#132F4C", size=18),
            hovertemplate="Actual %{y}<br>Predicted %{x}<br>Count %{z}<extra></extra>"
        ), row=row, col=col)

selected_cm_figure.update_xaxes(title_text="Predicted")
selected_cm_figure.update_yaxes(title_text="Actual", autorange="reversed")
selected_cm_figure.update_layout(template="plotly_white", height=690,
                                title="Test confusion matrices: default and validation-selected thresholds")
selected_cm_figure.show()

In [43]:
test_cms = {}
for name in model_names:
    test_cms[name] = [confusion_matrix(y_test, p_test[name] >= t, labels=[1, 0])
                      for t in thresholds]

threshold_cm_figure = make_subplots(rows=1, cols=3, subplot_titles=model_names,
                                   horizontal_spacing=0.10)
for col, name in enumerate(model_names, start=1):
    threshold_cm_figure.add_trace(go.Heatmap(
        z=test_cms[name][50], x=labels, y=labels,
        zmin=0, zmax=int(y_test.value_counts().max()), colorscale=colors,
        showscale=False, texttemplate="%{z}", textfont=dict(color="#132F4C", size=18),
        hovertemplate="Actual %{y}<br>Predicted %{x}<br>Count %{z}<extra></extra>"
    ), row=1, col=col)

steps = []
for i, t in enumerate(thresholds):
    steps.append(dict(method="update", label=f"{t:.2f}",
                      args=[{"z": [test_cms[name][i].tolist() for name in model_names]}]))

threshold_cm_figure.update_xaxes(title_text="Predicted")
threshold_cm_figure.update_yaxes(title_text="Actual", autorange="reversed")
threshold_cm_figure.update_layout(
    template="plotly_white", height=470,
    title="Test confusion matrices: explore every searched threshold",
    sliders=[dict(active=50, steps=steps, pad=dict(t=50),
                  currentvalue=dict(prefix="Common threshold = "))]
)
threshold_cm_figure.show()

In [44]:
metric_rows = []
for name in model_names:
    show_thresholds = sorted(set([0.10, 0.30, 0.50, 0.70, 0.90, best_thresholds[name]]))
    for t in show_thresholds:
        tp, fn, fp, tn = confusion_matrix(y_test, p_test[name] >= t, labels=[1, 0]).ravel()
        metric_rows.append({
            "Model": name, "Threshold": t, "Selected on validation": t == best_thresholds[name],
            "TP": tp, "FN": fn, "FP": fp, "TN": tn,
            "Accuracy": (tp + tn) / len(y_test), "Recall": tp / (tp + fn),
            "Precision": tp / (tp + fp) if tp + fp else 0,
            "Cost": cost_fp * fp + cost_fn * fn,
            "Log loss": log_loss(y_test, p_test[name])
        })

test_threshold_results = pd.DataFrame(metric_rows)
display(test_threshold_results.round(4))

,Model,Threshold,Selected on validation,TP,FN,FP,TN,Accuracy,Recall,Precision,Cost,Log loss
0,All 12 (unpenalized),0.10,False,4345,1532,17435,27758,0.6286,0.7393,0.1995,53837,0.3150
1,All 12 (unpenalized),0.30,False,1253,4624,2015,43178,0.8700,0.2132,0.3834,10669,0.3150
2,All 12 (unpenalized),0.50,False,242,5635,152,45041,0.8867,0.0412,0.6142,6091,0.3150
3,All 12 (unpenalized),0.70,False,11,5866,4,45189,0.8851,0.0019,0.7333,5878,0.3150
4,All 12 (unpenalized),0.78,True,0,5877,0,45193,0.8849,0.0000,0.0000,5877,0.3150
5,All 12 (unpenalized),0.90,False,0,5877,0,45193,0.8849,0.0000,0.0000,5877,0.3150
6,Ridge,0.10,False,4345,1532,17435,27758,0.6286,0.7393,0.1995,53837,0.3150
7,Ridge,0.30,False,1253,4624,2015,43178,0.8700,0.2132,0.3834,10669,0.3150
8,Ridge,0.50,False,242,5635,152,45041,0.8867,0.0412,0.6142,6091,0.3150
9,Ridge,0.70,False,11,5866,4,45189,0.8851,0.0019,0.7333,5878,0.3150
